# Sanity Check - Step 05: Bandpass Filter (1-40 Hz)

Überprüft:
- Filter erfolgreich angewendet
- Frequenzband korrekt (1-40 Hz)
- Power Spectral Density vor/nach Vergleich
- Amplituden reduziert

In [ ]:
import sys
from pathlib import Path
import mne
import numpy as np
import matplotlib.pyplot as plt

sys.path.append(str(Path.cwd().parent / 'eeg_pipeline'))
import config

print("Setup erfolgreich")

## 1. Before & After laden

In [ ]:
subject_id = config.SUBJECTS[0]
person = "P1"

before_path = config.OUTPUT_DIR / f"sub-{subject_id}_{person}_interpolated.fif"
after_path = config.OUTPUT_DIR / f"sub-{subject_id}_{person}_filtered.fif"

if before_path.exists() and after_path.exists():
    raw_before = mne.io.read_raw_fif(str(before_path), preload=False)
    raw_after = mne.io.read_raw_fif(str(after_path), preload=False)
    print(f"✓ Both files loaded for {person}")
else:
    print("✗ Files not found")

## 2. Metadata Überprüfung

In [ ]:
print(f"=== VORHER ===")
print(f"Sampling rate: {raw_before.info['sfreq']} Hz")
print(f"Kanäle: {len(raw_before.ch_names)}")
print(f"Dauer: {raw_before.times[-1]:.2f}s")

print(f"\n=== NACHHER (gefiltert) ===")
print(f"Sampling rate: {raw_after.info['sfreq']} Hz")
print(f"Kanäle: {len(raw_after.ch_names)}")
print(f"Dauer: {raw_after.times[-1]:.2f}s")

print(f"\n=== VALIDIERUNGEN ===")
if len(raw_before.ch_names) == len(raw_after.ch_names):
    print(f"✓ Kanal-Anzahl erhalten")
if raw_before.info['sfreq'] == raw_after.info['sfreq']:
    print(f"✓ Sampling rate erhalten")
if raw_before.n_times == raw_after.n_times:
    print(f"✓ Sample-Anzahl erhalten")

## 3. Amplituden-Vergleich

In [ ]:
eeg_picks = mne.pick_types(raw_before.info, eeg=True)

# Get small sample for comparison
t_end = min(60, raw_before.times[-1])
t_idx_end = int(t_end * raw_before.info['sfreq'])

data_before = raw_before.get_data(picks=eeg_picks[0:1], start=0, stop=t_idx_end)
data_after = raw_after.get_data(picks=eeg_picks[0:1], start=0, stop=t_idx_end)

std_before = np.std(data_before)
std_after = np.std(data_after)
reduction = (1 - std_after/std_before) * 100

print(f"Amplituden (erstes EEG-Kanal, erste 60s):")
print(f"  VORHER: {std_before:.6f} µV (Std)")
print(f"  NACHHER: {std_after:.6f} µV (Std)")
print(f"  Reduktion: {reduction:.1f}%")

## 4. PSD Vergleich

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# VORHER
raw_before_eeg = raw_before.copy().pick_types(eeg=True)
raw_before_eeg.plot_psd(fmax=60, ax=axes[0], show=False)
axes[0].axvline(x=config.FREQ_LOWER, color='red', linestyle='--', label=f'Filter: {config.FREQ_LOWER} Hz')
axes[0].axvline(x=config.FREQ_UPPER, color='red', linestyle='--', label=f'Filter: {config.FREQ_UPPER} Hz')
axes[0].set_title('BEFORE: Power Spectral Density')
axes[0].legend()

# NACHHER
raw_after_eeg = raw_after.copy().pick_types(eeg=True)
raw_after_eeg.plot_psd(fmax=60, ax=axes[1], show=False)
axes[1].axvline(x=config.FREQ_LOWER, color='red', linestyle='--', label=f'Filter: {config.FREQ_LOWER} Hz')
axes[1].axvline(x=config.FREQ_UPPER, color='red', linestyle='--', label=f'Filter: {config.FREQ_UPPER} Hz')
axes[1].set_title(f'AFTER: PSD (Filtered {config.FREQ_LOWER}-{config.FREQ_UPPER} Hz)')
axes[1].legend()

plt.tight_layout()
plt.savefig(config.QC_DIR / f'sub-{subject_id}_{person}_filter_psd_comparison.png', dpi=100, bbox_inches='tight')
plt.show()
print(f"✓ PSD plot saved")